In [38]:
import os
import time
import json
import requests
import pandas as pd
import argparse
import psycopg2

# ----------------------------
# API Ice and Fire
# ----------------------------
BASE_API = "https://www.anapioficeandfire.com/api"
BOOKS_ENDPOINT = f"{BASE_API}/books"
HOUSES_ENDPOINT = f"{BASE_API}/houses"

def fetch_all_books(page_size: int = 20) -> pd.DataFrame:
    page = 1
    all_books = []
    while True:
        resp = requests.get(BOOKS_ENDPOINT, params={"page": page, "pageSize": page_size})
        resp.raise_for_status()
        data = resp.json()
        if not data:
            break
        all_books.extend(data)
        if len(data) < page_size:
            break
        page += 1
        time.sleep(0.1)
    return pd.DataFrame(all_books)

def fetch_all_houses(page_size: int = 50) -> pd.DataFrame:
    page = 1
    all_houses = []
    while True:
        resp = requests.get(HOUSES_ENDPOINT, params={"page": page, "pageSize": page_size})
        resp.raise_for_status()
        data = resp.json()
        if not data:
            break
        all_houses.extend(data)
        if len(data) < page_size:
            break
        page += 1
        time.sleep(0.1)
    return pd.DataFrame(all_houses)

def fetch_houses_with_titles_api(use_api_filter: bool = True, page_size: int = 50) -> pd.DataFrame:
    """
    Пробуем фильтр hasTitles через API. Если данные пустые или фильтр не поддерживается,
    возвращаем DataFrame, возможно пост-фильтруя по words (если они есть).
    """
    page = 1
    all_houses = []
    api_filter_used = False

    while True:
        params = {"page": page, "pageSize": page_size}
        if use_api_filter:
            params["hasTitles"] = "true"
            api_filter_used = True
        resp = requests.get(HOUSES_ENDPOINT, params=params)
        resp.raise_for_status()
        data = resp.json()
        if not data:
            break
        all_houses.extend(data)
        if len(data) < page_size:
            break
        page += 1
        time.sleep(0.1)

    df = pd.DataFrame(all_houses)

    # Пост-фильтрация по words, если API вернул данные без девизов
    if 'words' in df.columns:
        df_with_words = df[df['words'].astype(str).str.strip().astype(bool)]
        if not df_with_words.empty:
            return df_with_words

    if api_filter_used:
        return df

    if 'words' in df.columns:
        return df[df['words'].astype(str).str.strip().astype(bool)]

    return df

# ----------------------------
# RNAcentral Public PostgreSQL
# ----------------------------
# Публичные параметры подключения (без пароля)
RNC_HOST = "hh-pgsql-public.ebi.ac.uk"
RNC_PORT = 5432
RNC_DATABASE = "pfmegrnargs"
RNC_USER = "reader"
RNAC_PASSWORD = "NWDMCE5xdipIjRrp"

def get_rnc_password() -> str:
    return os.environ.get("RNAC_PASSWORD") or RNAC_PASSWORD

def get_rnc_connection():
    password = get_rnc_password()
    if not password:
        raise RuntimeError("RNAC_PASSWORD не задана. Установите переменную окружения RNAC_PASSWORD.")
    return psycopg2.connect(
        host=RNC_HOST,
        database=RNC_DATABASE,
        user=RNC_USER,
        password=password,
        port=RNC_PORT
    )

def fetch_10_rows_from_rnc_database() -> pd.DataFrame:
    conn = get_rnc_connection()
    try:
        cur = conn.cursor()
        cur.execute("SELECT * FROM rnc_database LIMIT 10;")
        rows = cur.fetchall()
        colnames = [desc[0] for desc in cur.description]
        df = pd.DataFrame(rows, columns=colnames)
        cur.close()
        return df
    finally:
        conn.close()

def fetch_selected_columns_from_rnc_database() -> pd.DataFrame:
    conn = get_rnc_connection()
    try:
        cur = conn.cursor()
        cur.execute("SELECT display_name, num_sequences, num_organisms, url FROM rnc_database LIMIT 10;")
        rows = cur.fetchall()
        colnames = [desc[0] for desc in cur.description]
        df = pd.DataFrame(rows, columns=colnames)
        cur.close()
        return df
    finally:
        conn.close()

# ----------------------------
# Утилиты
# ----------------------------
def save_csv(df: pd.DataFrame, path: str):
    df.to_csv(path, index=False, encoding='utf-8')
    print(f"Сохранено {len(df)} строк в {path}")

# ----------------------------
# Главная точка входа
# ----------------------------
def main():
    parser = argparse.ArgumentParser(description="Объединённое задание: API Ice and Fire + RNAcentral PostgreSQL")
    parser.add_argument("--mode", choices=[
        "api_books",
        "api_houses",
        "api_houses_with_titles",
        "db_10",
        "db_selected",
        "all"
    ], default="all", help="Какой раздел выполнить")
    parser.add_argument("--save-csv", action="store_true", help="Сохранить результаты в CSV")
    args, _ = parser.parse_known_args()

    results = {}

    # 1) API: книги
    if args.mode in ("api_books", "all"):
        print("Загрузка книг из Ice and Fire API...")
        try:
            books_df = fetch_all_books(page_size=20)
        except Exception as e:
            print("Ошибка загрузки книг:", e)
            books_df = pd.DataFrame()
        results["api_books"] = books_df
        print(f"Книги: {len(books_df)}")
        if args.save_csv and not books_df.empty:
            save_csv(books_df, "books.csv")

    # 2) API: дома
    if args.mode in ("api_houses", "all"):
        print("Загрузка домов (Houses) из Ice and Fire API...")
        try:
            houses_df = fetch_all_houses(page_size=50)
        except Exception as e:
            print("Ошибка загрузки домов:", e)
            houses_df = pd.DataFrame()
        results["api_houses"] = houses_df
        print(f"Дома: {len(houses_df)}")
        if args.save_csv and not houses_df.empty:
            save_csv(houses_df, "houses.csv")

    # 3) API: дома с девизами (hasTitles)
    if args.mode in ("api_houses_with_titles", "all"):
        print("Загрузка домов с девизами/названиями (hasTitles) из Ice and Fire API...")
        try:
            houses_titles_df = fetch_houses_with_titles_api(use_api_filter=True, page_size=50)
        except Exception as e:
            print("Ошибка загрузки домов с titles:", e)
            houses_titles_df = pd.DataFrame()
        results["api_houses_with_titles"] = houses_titles_df
        print(f"Дома с титулами/девизами: {len(houses_titles_df)}")
        if args.save_csv and not houses_titles_df.empty:
            save_csv(houses_titles_df, "houses_with_titles.csv")

    # 4) RNAcentral: БД
    if args.mode in ("db_10", "all"):
        print("Пытаемся загрузить 10 строк из rnc_database...")
        try:
            df10 = fetch_10_rows_from_rnc_database()
            results["db_10"] = df10
            print(f"Получено {len(df10)} строк из rnc_database")
            if args.save_csv and not df10.empty:
                save_csv(df10, "db_rnc_10_rows.csv")
        except Exception as e:
            print("Не удалось подключиться к RNAcentral DB или выполнить запрос:", e)
            df10 = pd.DataFrame()
            results["db_10"] = df10

        print("Пытаемся получить 10 строк с выбранными столбцами...")
        try:
            df_selected = fetch_selected_columns_from_rnc_database()
            results["db_selected"] = df_selected
            print(f"Получено {len(df_selected)} строк с выбранными столбцами")
            if args.save_csv and not df_selected.empty:
                save_csv(df_selected, "db_rnc_selected_columns.csv")
        except Exception as e:
            print("Не удалось получить выбранные столбцы:", e)
            df_selected = pd.DataFrame()
            results["db_selected"] = df_selected

    print("Готово.")
    for key, df in results.items():
        if isinstance(df, pd.DataFrame):
            print(f"{key}: размер = {len(df)}")
        else:
            print(f"{key}: type={type(df)}")

if __name__ == "__main__":
    main()

Загрузка книг из Ice and Fire API...
Книги: 12
Загрузка домов (Houses) из Ice and Fire API...
Дома: 444
Загрузка домов с девизами/названиями (hasTitles) из Ice and Fire API...
Дома с титулами/девизами: 41
Пытаемся загрузить 10 строк из rnc_database...
Получено 10 строк из rnc_database
Пытаемся получить 10 строк с выбранными столбцами...
Получено 10 строк с выбранными столбцами
Готово.
api_books: размер = 12
api_houses: размер = 444
api_houses_with_titles: размер = 41
db_10: размер = 10
db_selected: размер = 10


In [ ]:
from google.colab import drive
drive.mount('/content/drive')